In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent.parent))

In [3]:
import pandas as pd
from questionnaire_processing import (
    load_named_csvs,
    clean_phq8_dataframe,
    select_model_ready_columns,
    aggregate_participant_labels,
)

In [6]:
data_folder = Path("../datasets/RADAR-MDD/speech-features-for-prediction")

filenames = [
    "RADAR-MDD-Anna-Finch-RADAR-MDD-CIBER-s1-07-08-2025.csv",
    "RADAR-MDD-Anna-Finch-RADAR-MDD-IISPV-s1-07-08-2025.csv",
    "RADAR-MDD-Anna-Finch-RADAR-MDD-KCL-s1-07-08-2025.csv",
    "RADAR-MDD-Anna-Finch-RADAR-MDD-VUmc-s1-07-08-2025.csv",
    "phq8_data.csv",
]

loaded = load_named_csvs(data_folder, filenames)

In [7]:
for name, df in loaded.items():
    print(f"{name}: {df.shape}")

RADAR-MDD-Anna-Finch-RADAR-MDD-CIBER-s1-07-08-2025.csv: (2086, 33)
RADAR-MDD-Anna-Finch-RADAR-MDD-IISPV-s1-07-08-2025.csv: (138, 33)
RADAR-MDD-Anna-Finch-RADAR-MDD-KCL-s1-07-08-2025.csv: (8517, 33)
RADAR-MDD-Anna-Finch-RADAR-MDD-VUmc-s1-07-08-2025.csv: (3153, 33)
phq8_data.csv: (13742, 48)


In [8]:
phq8_raw = loaded["phq8_data.csv"]
phq8_raw.head()

,participant_name,file_name,file_size,file_modified_on,project_id,user_id,source_id,time,time_completed_utc,time_completed_local,...,question_id6,value6,start_time6,end_time6,question_id7,value7,start_time7,end_time7,created_at,updated_at
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,394,1556732136,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,1546856806,1546857094,2019-01-07 11:31:34,...,phq8_7,2,1546857072,1546857084,phq8_8,0,1546857084,1546857094,2020-02-07 06:34:11,2020-02-07 06:34:11
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,396,1556732146,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,1548100687,1548100775,2019-01-21 20:59:35,...,phq8_7,1,1548100744,1548100766,phq8_8,2,1548100766,1548100775,2020-02-07 06:34:11,2020-02-07 06:34:11
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,394,1556732150,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,1549282178,1549282257,2019-02-04 13:10:57,...,phq8_7,2,1549282238,1549282245,phq8_8,2,1549282245,1549282257,2020-02-07 06:34:11,2020-02-07 06:34:11
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,390,1556732169,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,1551687016,1551687094,2019-03-04 09:11:34,...,phq8_7,2,1551687075,1551687083,phq8_8,1,1551687083,1551687094,2020-02-07 06:34:11,2020-02-07 06:34:11
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,391,1556732169,RADAR-MDD-CIBER-s1,007751c5-d7ad-4bec-a58f-abf32500e2ae,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,1554107096,1554107186,2019-04-01 10:26:26,...,phq8_7,1,1554107167,1554107174,phq8_8,1,1554107174,1554107186,2020-02-07 06:34:11,2020-02-07 06:34:11


In [9]:
phq8_raw.columns.tolist()

['participant_name',
 'file_name',
 'file_size',
 'file_modified_on',
 'project_id',
 'user_id',
 'source_id',
 'time',
 'time_completed_utc',
 'time_completed_local',
 'time_interval',
 'time_notification',
 'name',
 'version',
 'question_id0',
 'value0',
 'start_time0',
 'end_time0',
 'question_id1',
 'value1',
 'start_time1',
 'end_time1',
 'question_id2',
 'value2',
 'start_time2',
 'end_time2',
 'question_id3',
 'value3',
 'start_time3',
 'end_time3',
 'question_id4',
 'value4',
 'start_time4',
 'end_time4',
 'question_id5',
 'value5',
 'start_time5',
 'end_time5',
 'question_id6',
 'value6',
 'start_time6',
 'end_time6',
 'question_id7',
 'value7',
 'start_time7',
 'end_time7',
 'created_at',
 'updated_at']

In [10]:
phq8_clean = clean_phq8_dataframe(phq8_raw)
phq8_clean.shape

(13742, 52)

In [11]:
phq8_clean[[
    "participant_id",
    "project_id",
    "questionnaire_name",
    "completed_at",
    "phq8_score",
    "severity"
]].head()

,participant_id,project_id,questionnaire_name,completed_at,phq8_score,severity
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,PHQ8,2019-01-07 11:31:34,9,mild
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,PHQ8,2019-01-21 20:59:35,11,moderate
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,PHQ8,2019-02-04 13:10:57,12,moderate
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,PHQ8,2019-03-04 09:11:34,11,moderate
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,PHQ8,2019-04-01 10:26:26,11,moderate


In [12]:
phq8_model = select_model_ready_columns(phq8_clean)
phq8_model.to_csv("../data/processed/questionnaire_phq8_clean.csv", index=False)
phq8_model.head()

,participant_id,project_id,site,questionnaire_name,version,completed_at,time_completed_utc,phq8_score,severity,file_name,source_id,created_at,updated_at
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,RADAR-MDD-CIBER-s1,PHQ8,0.2.0,2019-01-07 11:31:34,2019-01-07 10:31:34,9,mild,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,2020-02-07 06:34:11,2020-02-07 06:34:11
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,RADAR-MDD-CIBER-s1,PHQ8,0.2.0,2019-01-21 20:59:35,2019-01-21 19:59:35,11,moderate,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,2020-02-07 06:34:11,2020-02-07 06:34:11
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,RADAR-MDD-CIBER-s1,PHQ8,0.2.0,2019-02-04 13:10:57,2019-02-04 12:10:57,12,moderate,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,2020-02-07 06:34:11,2020-02-07 06:34:11
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,RADAR-MDD-CIBER-s1,PHQ8,0.2.0,2019-03-04 09:11:34,2019-03-04 08:11:34,11,moderate,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,2020-02-07 06:34:11,2020-02-07 06:34:11
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,RADAR-MDD-CIBER-s1,PHQ8,0.2.0,2019-04-01 10:26:26,2019-04-01 08:26:26,11,moderate,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,2020-02-07 06:34:11,2020-02-07 06:34:11


In [13]:
participant_labels_latest = aggregate_participant_labels(phq8_model, strategy="latest")
participant_labels_mean = aggregate_participant_labels(phq8_model, strategy="mean")
participant_labels_max = aggregate_participant_labels(phq8_model, strategy="max")

In [14]:
participant_labels_latest.to_csv("../data/processed/participant_labels_latest.csv", index=False)
participant_labels_mean.to_csv("../data/processed/participant_labels_mean.csv", index=False)
participant_labels_max.to_csv("../data/processed/participant_labels_max.csv", index=False)

In [15]:
phq8_model["phq8_score"].describe()

count    13742.000000
mean        10.111410
std          6.193343
min          0.000000
25%          5.000000
50%          9.000000
75%         15.000000
max         24.000000
Name: phq8_score, dtype: float64

In [16]:
phq8_model["severity"].value_counts(dropna=False)

severity
mild                 4168
moderate             3176
minimal              2832
moderately_severe    2257
severe               1309
Name: count, dtype: int64

In [17]:
phq8_model["participant_id"].nunique()

595

In [18]:
phq8_model["project_id"].value_counts(dropna=False)

project_id
RADAR-MDD-KCL-s1      7716
RADAR-MDD-CIBER-s1    3268
RADAR-MDD-VUmc-s1     2605
RADAR-MDD-IISPV-s1     153
Name: count, dtype: int64

In [21]:
phq8_model.groupby("participant_id").size().describe()

count    595.000000
mean      23.095798
std       14.381616
min        1.000000
25%       11.000000
50%       22.000000
75%       33.000000
max       59.000000
dtype: float64

In [19]:
phq8_model.loc[
    (phq8_model["phq8_score"] < 0) | (phq8_model["phq8_score"] > 24),
    ["participant_id", "phq8_score"]
]

,participant_id,phq8_score


In [20]:
for name in filenames[:-1]:
    df = loaded[name]
    print("\n", name)
    print(df.head())
    print(df.columns.tolist())


 RADAR-MDD-Anna-Finch-RADAR-MDD-CIBER-s1-07-08-2025.csv
                             File                            Patient_id  \
0  20200929_0800-scripted-1-1.wav  007751c5-d7ad-4bec-a58f-abf32500e2ae   
1  20200610_1100-scripted-2-1.wav  007751c5-d7ad-4bec-a58f-abf32500e2ae   
2  20200901_1100-unscripted-1.wav  007751c5-d7ad-4bec-a58f-abf32500e2ae   
3  20210315_2000-scripted-1-1.wav  007751c5-d7ad-4bec-a58f-abf32500e2ae   
4  20210215_1100-scripted-2-1.wav  007751c5-d7ad-4bec-a58f-abf32500e2ae   

     Dataset Language        Task        Date   Age  Gender  Education_Years  \
0  RADAR-MDD  Spanish    Scripted  2020-09-29  55.0     1.0             22.0   
1  RADAR-MDD  Spanish    Scripted  2020-06-10  55.0     1.0             22.0   
2  RADAR-MDD  Spanish  Unscripted  2020-09-01  55.0     1.0             22.0   
3  RADAR-MDD  Spanish    Scripted  2021-03-15  55.0     1.0             22.0   
4  RADAR-MDD  Spanish    Scripted  2021-02-15  55.0     1.0             22.0   

   Height  

In [22]:
participant_labels_latest = aggregate_participant_labels(
    phq8_model,
    strategy="latest"
)

participant_labels_latest.head()

,participant_id,project_id,site,questionnaire_name,version,completed_at,time_completed_utc,phq8_score,severity,file_name,source_id,created_at,updated_at
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,RADAR-MDD-CIBER-s1,RADAR-MDD-CIBER-s1,PHQ8,0.5.1,2021-03-29 11:17:15,2021-03-29 09:17:15,17,moderately_severe,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,20555f76-dbd2-455b-8ad9-bbe3fb757fdb,2021-05-05 17:22:49,2021-05-05 17:22:49
1,00bf567f-30ef-4317-9c3d-f62a37bcce27,RADAR-MDD-VUmc-s1,RADAR-MDD-VUmc-s1,PHQ8,0.4.2,2020-09-16 14:44:33,2020-09-16 12:44:33,11,moderate,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,a1cf76bc-aa98-4a5a-9711-89d98172f913,2020-09-16 17:21:31,2020-09-16 17:21:31
2,00d1d60a-15cf-481b-8264-91705b6f8d97,RADAR-MDD-KCL-s1,RADAR-MDD-KCL-s1,PHQ8,0.4.3,2021-03-24 09:31:51,2021-03-24 09:31:51,12,moderate,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,f8228da7-54ee-4654-bf7b-6cca782a7fe0,2021-05-05 17:21:23,2021-05-05 17:21:23
3,0104dfff-4dcd-48ff-b912-51362f098ed0,RADAR-MDD-VUmc-s1,RADAR-MDD-VUmc-s1,PHQ8,0.4.2,2021-03-24 09:43:30,2021-03-24 08:43:30,12,moderate,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,0938bf81-1edb-40a5-8720-fe11567532b3,2021-05-05 17:22:32,2021-05-05 17:22:32
4,01318193-bb33-4c1b-b1bf-d316a014b5ad,RADAR-MDD-CIBER-s1,RADAR-MDD-CIBER-s1,PHQ8,0.3.1,2020-02-28 09:58:51,2020-02-28 08:58:51,8,mild,/mnt/pool0/RADAR-CNS/HDFS_CSV/output-gzip/RADA...,be774af6-8d97-4546-8f85-282df178d581,2020-02-28 17:27:47,2020-02-28 17:27:47


In [23]:
import pandas as pd

phq8 = pd.read_csv("../data/processed/questionnaire_phq8_clean.csv")
print(phq8.shape)
print(phq8.columns.tolist())
print(phq8[["participant_id", "completed_at", "phq8_score", "severity"]].head())

(13742, 13)
['participant_id', 'project_id', 'site', 'questionnaire_name', 'version', 'completed_at', 'time_completed_utc', 'phq8_score', 'severity', 'file_name', 'source_id', 'created_at', 'updated_at']
                         participant_id         completed_at  phq8_score  \
0  007751c5-d7ad-4bec-a58f-abf32500e2ae  2019-01-07 11:31:34           9   
1  007751c5-d7ad-4bec-a58f-abf32500e2ae  2019-01-21 20:59:35          11   
2  007751c5-d7ad-4bec-a58f-abf32500e2ae  2019-02-04 13:10:57          12   
3  007751c5-d7ad-4bec-a58f-abf32500e2ae  2019-03-04 09:11:34          11   
4  007751c5-d7ad-4bec-a58f-abf32500e2ae  2019-04-01 10:26:26          11   

   severity  
0      mild  
1  moderate  
2  moderate  
3  moderate  
4  moderate  
